In [2]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import AutoEncoder
import pydicom

pydicom.config.settings.reading_validation_mode = pydicom.config.WARN 


In [5]:

df_data = pd.read_csv("./data/labels_duke_nifti.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
                            data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
                        )

In [11]:

transforms = Compose([
    LoadImaged(keys=["nifti"], ensure_channel_first=True),                
    Orientationd(keys="nifti", axcodes="RAS"),            
    Spacingd(keys="nifti", pixdim=(1.0, 1.0, 1.0),        
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                       
        keys="nifti", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="nifti", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="nifti", prob=0.5, spatial_axis=0),    
    RandRotate90d(keys="nifti", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="nifti", prob=0.2),
    EnsureTyped(keys="nifti"),
])



/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [12]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


ds = CacheDataset(data, transform=transforms, cache_rate=1.0, num_workers=16)


Loading dataset: 100%|██████████| 917/917 [08:29<00:00,  1.80it/s]


In [61]:
from monai.data import ThreadDataLoader

loader = ThreadDataLoader(ds, batch_size=100, shuffle=True, num_workers=0)

device = torch.device("cuda")
model = AutoEncoder(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2, 2),
    inter_channels=(256,),        
    inter_dilations=(1,),
    num_res_units=2,
).to(device)

opt = torch.optim.Adam(model.parameters(), 1e-3)
loss_fn = torch.nn.MSELoss()

In [62]:
model.train()
for epoch in range(50):
    for batch in loader:
        x = batch["nifti"].to(device)
        opt.zero_grad()
        recon = model(x)
        loss = loss_fn(recon, x)
        loss.backward()
        opt.step()
    print(epoch, loss.item())

0 0.07670256495475769
1 0.052723776549100876
2 0.0343034453690052
3 0.024526238441467285
4 0.018827611580491066
5 0.014510570093989372
6 0.011998913250863552
7 0.011783743277192116
8 0.011949706822633743
9 0.010431909002363682
10 0.011011296883225441
11 0.00869440846145153
12 0.008783109486103058
13 0.009248818270862103
14 0.007668597623705864
15 0.008470132015645504
16 0.00792043935507536
17 0.007984895259141922
18 0.007128623779863119
19 0.007605747319757938
20 0.006632677279412746
21 0.0069667380303144455
22 0.00645699305459857
23 0.006621767301112413
24 0.005749020725488663
25 0.005139710381627083
26 0.006506301928311586
27 0.005293348338454962
28 0.005773389246314764
29 0.00635567307472229
30 0.006956569384783506
31 0.005584320984780788
32 0.005452804733067751
33 0.0061511509120464325
34 0.005329515319317579
35 0.006214114837348461
36 0.005990372039377689
37 0.006462981458753347
38 0.005635829176753759
39 0.005134408827871084
40 0.004915154073387384
41 0.004658101592212915
42 0.00

In [58]:
print("peak allocated:", torch.cuda.max_memory_allocated() / 1e9, "GB")
print("peak reserved: ", torch.cuda.max_memory_reserved() / 1e9, "GB")
print(torch.cuda.memory_stats()["num_alloc_retries"])

peak allocated: 14.956973056 GB
peak reserved:  18.209570816 GB
0


In [66]:
import gc, torch

#del model, opt, loader   # whatever holds the big objects
gc.collect()                              # free Python/RAM references
torch.cuda.empty_cache()                  # return cached VRAM to the GPU
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_peak_memory_stats()
# ... run a few training steps ...
